In [1]:
import numpy as np
import pandas as pd
import time
import os
import random
import joblib

import elkai
import tsplib95
from scipy.spatial.distance import cdist
from scipy.stats import skew, kurtosis
from scipy.sparse.csgraph import minimum_spanning_tree

from mealpy import GA, SA, PSO, ACOR
from mealpy.utils.problem import Problem
from mealpy.utils.space import FloatVar

class TSPProblem(Problem):
    def __init__(self, D, **kwargs):
        self.D = D
        self.n = len(D)
        bounds = FloatVar(lb=[0.0] * self.n, ub=[self.n - 1.0] * self.n)
        super().__init__(
            bounds=bounds,
            minmax="min",
            **kwargs
        )

    def obj_func(self,x):
        tour = np.argsort(x).tolist()
        cost = sum(self.D[tour[i]][tour[i+1]] for i in range(self.n - 1))
        cost += self.D[tour[-1]][tour[0]]
        return cost

def two_opt(tour, D):
    n = len(tour)
    improved = True
    while improved:
        improved = False
        for i in range(1, n-1):
            for j in range(i+1, n):
                a, b = tour[i-1], tour[i]
                c, d = tour[j], tour[(j+1) % n]
                if D[a][b] + D[c][d] > D[a][c] + D[b][d]:
                    tour[i:j+1] = tour[i:j+1][::-1]
                    improved = True
    return tour


In [2]:
def run_mealpy(model,D):
    problem = TSPProblem(D=D, log_to=None)
    t0 = time.time()
    model.solve(problem)
    runtime = time.time() - t0
    best_x = model.g_best.solution
    tour = np.argsort(best_x).tolist()
    tour = two_opt(tour, D)
    cost = sum(D[tour[i]][tour[i+1]] for i in range(len(tour)-1)) + D[tour[-1]][tour[0]]
    history = model.history.list_global_best_fit
    return cost, runtime, history

def run_lk(D):
    D_int = (D * 1000).astype(int).tolist()
    t0 = time.time()
    tour = elkai.solve_int_matrix(D_int)
    runtime = time.time() - t0
    cost = sum(D[tour[i]][tour[i+1]] for i in range(len(tour) - 1)) + D[tour[-1]][tour[0]]
    return cost, runtime, None

In [3]:
def get_algo_pool(n):
    if n<=30:
        epochs, pop = 200,100
    elif n<=75:
        epochs, pop = 300,100
    elif n<=100:
        epochs, pop = 400,100
    elif n<=150:
        epochs, pop = 500,150
    else:
        epochs, pop = 800,200
        
    return {
        "OGA": lambda D, e=epochs, p=pop: run_mealpy(GA.OriginalGA(epoch=e, pop_size=p), D),
        "SA":  lambda D, e=epochs: run_mealpy(SA.OriginalSA(epoch=e, pop_size=10), D),
        "LK":  lambda D: run_lk(D),
        "PSO": lambda D, e=epochs, p=pop: run_mealpy(PSO.OriginalPSO(epoch=e, pop_size=p), D),
        
    }


In [4]:
def compute_distance_matrix(coords, distance_type="euclidean"):
    n = len(coords)
    D = np.zeros((n,n))

    if distance_type == "euclidean":
        D = cdist(coords, coords, metrix='euclidean')

    elif distance_type == "manhattan":
        D = cdist(coords, coords, metric="cityblock")

    elif distance_type == "chebyshev":
        D = cdist(coords, coords, metric="chebyshev")

    elif distance_type == "ceil_euclidean":
        D = np.ceil(cdist(coords, coords, metric="euclidean"))

    else:
        raise ValueError(f"Unsupported distance type: {distance_type}")
        
    np.fill_diagonal(D, 0)
    return D

DISTANCE_TYPE_ENCODING = {
    "euclidean": 0,
    "manhattan": 1,
    "chebyshev": 2,
    "ceil_euclidean": 3
}

In [5]:
def generate_instance(n, distribution, seed, distance_type="euclidean"):
    rng = np.random.default_rng(seed)

    if distribution == "uniform":
        coords = rng.uniform(0, 1000, (n,2))

    elif distribution == "clustered":
        centers = rng.uniform(100, 900, (max(2, n//10),2))
        coords = np.array([centers[i % max(2, n//10)] + rng.normal(0,30,2) for i in range(n)])

    elif distribution == "grid":
        side = int(np.ceil(np.sqrt(n)))
        coords = np.array([[x,y] for x in range(side) for y in range(side)])[:n] * (1000/side)

    elif distribution == "diagonal":
        t = rng.uniform(0, 1000, n)
        noise = rng.normal(0,20,(n,2))
        coords = np.column_stack([t, t]) + noise

    D = cdist(coords, coords, distance_type)
    return coords, D

In [6]:
def load_tsplib(name, folder="./tsplib"):
    problem = tsplib95.load(os.path.join(folder, f"{name}.tsp"))
    n = problem.dimension
    nodes = list(problem.get_nodes())

    D = np.zeros((n, n))
    for i in nodes:
        for j in nodes:
            D[i-1][j-1] = problem.get_weight(i, j)

    coords = None
    if problem.node_coords:
        coords = np.array([problem.node_coords[i] for i in nodes])

    return coords, D

In [7]:
def extract_features(coords, D):
    n = len(D)
    upper = D[np.triu_indices(n, k=1)]

    D_nn = D.copy()
    np.fill_diagonal(D_nn, np.inf)
    nn = D_nn.min(axis=1)

    mst_arr = minimum_spanning_tree(D).toarray()
    mst_vals = mst_arr[mst_arr > 0]

    features = {
        "n_cities": n,
        "avg_edge": upper.mean(),
        "std_edge": upper.std(),
        "cv_edge": upper.std() / upper.mean(),
        "skew_edge": float(skew(upper)),
        "kurtosis_edge": float(kurtosis(upper)),
        "min_edge": upper.min(),
        "max_edge": upper.max(),

        "avg_nn": nn.mean(),
        "std_nn": nn.std(),
        "cv_nn": nn.std() / nn.mean(),
        "skew_nn": float(skew(nn)),

        "mst_cost": mst_vals.sum(),
        "avg_mst_edge": mst_vals.mean(),
        "std_mst_edge": mst_vals.std(),
        "mst_nn_ratio": mst_vals.mean() / nn.mean()
    }

    if coords is not None:
        centroid = coords.mean(axis=0)
        centroid_dists = np.linalg.norm(coords - centroid, axis=1)

        x_range = coords[:, 0].max() - coords[:, 0].min()
        y_range = coords[:, 1].max() - coords[:, 1].min()

        features.update({
            "avg_centroid_dist": centroid_dists.mean(),
            "std_centroid_dist": centroid_dists.std(),
            "cv_centroid_dist": centroid_dists.std() / centroid_dists.mean(),
            "bbox_area": x_range * y_range,
            "bbox_ratio": x_range / (y_range +1e-9)
        })

    else:
        for col in ["avg_centroid_dist","std_centroid_dist",
                    "cv_centroid_dist","bbox_area","bbox_ratio"]:
            features[col] = np.nan
        
    return features


In [8]:
def solve_concorde(coords):
    try:
        from concorde.tsp import TSPSolver
        solver = TSPSolver.from_data(
            coords[:, 0].tolist(),
            coords[:, 1].tolist(),
            norm = "EUC_2D"
        )
        solution = solve.solver(verbose=False)
        tour = list(solution.tour)
        D = cdist(coords, coords)
        cost = sum(D[tour[i]][tour[i_1]] for i in range(len(tour)-1)) + D[tour[-1]][tour[0]]
        
        return cost
    except ImportError:
        return None

In [9]:
def run_all(D, coords=None, n_runs=3, use_concorde=False):
    n = len(D)
    results = {}
    pool = get_algo_pool(n)

    for name, algo_fn  in pool.items():
        run_costs = []
        run_times = []

        for _ in range(n_runs):
            cost, runtime, _ = algo_fn(D)
            run_costs.append(cost)
            run_times.append(runtime)
        results[name] = {
            "best_cost": min(run_costs),
            "avg_cost": np.mean(run_costs),
            "std_cost": np.std(run_costs),
            "avg_time": np.mean(run_times),
        }

    optimal = None
    if use_concorde and coords is not None and len(D) <= 150:
        optimal = solve_concorde(coords)

    return results, optimal

In [10]:
def validate_tour(tour, n):
    if len(tour)!=n:
        return False, f"Tour length {len(tour)} does not match n={n}"

    expected = set(range(n))
    actual = set(tour)

    if actual!=expected:
        missing = expected - actual
        extra = actual - expected
        msh = ""
        if missing:
            msg += f"Missing cities: {missing}"
        if extra:
            msg += f"Invalid cities: {extra}"
        return False, msg
    if len(tour) != len(set(tour)):
        duplicates = [c for c in tour if tour.count(c) > 1]
        return False, f"Duplicate cities: {set(duplicates)}"
    
    return True, "Valid"

In [11]:
def build_row(coords, D, source, label, n_runs=3, use_concorde=False):
    features = extract_features(coords, D)
    algo_results, optimal = run_all(
        D, coords, n_runs=n_runs, use_concorde=use_concorde
    )

    best_algo = min(algo_results, key=lambda k: algo_results[k]["best_cost"])

    row = {
        **features,
        "source": source,
        "label": label,
        "best_algo": best_algo,
        "optimal": optimal,
    }

    for algo, metrics in algo_results.items():
        for metric, val in metrics.items():
            row[f"{algo}_{metric}"] = val

        if optimal:
            row[f"{algo}_gap"] = (
                (algo_results[algo]["best_cost"] - optimal / optimal)
            )
        return row
    
def build_dataset(configs, source="generated", use_concorde=False):
    rows=[]
    for i, cfg in enumerate(configs):
        if source == "generated":
            coords, D = generate_instance(
                cfg["n"], cfg["distribution"], cfg["seed"], cfg.get("distance_type","euclidean")
            )
            label = f"{cfg['distribution']}_{cfg['n']}_{cfg['seed']}"
        else:
            coords, D = load_tsplib(cfg["name"])
            label = cfg["name"]
        
        row = build_row(
            coords, D,
            source=source,
            label=label,
            use_concorde=use_concorde
        )
        rows.append(row)
        print(f"[{i+1}/{len(configs)}] {label} -> best: {row['best_algo']}")
    
    return pd.DataFrame(rows)



# Validation


In [18]:
def run_mealpy_val(model,D):
    problem = TSPProblem(D=D, log_to=None)
    t0 = time.time()
    model.solve(problem)
    runtime = time.time() - t0
    best_x = model.g_best.solution
    tour = np.argsort(best_x).tolist()
    tour = two_opt(tour, D)
    cost = sum(D[tour[i]][tour[i+1]] for i in range(len(tour)-1)) + D[tour[-1]][tour[0]]
    history = model.history.list_global_best_fit
    return cost, runtime, history, tour

def run_lk_val(D):
    D_int = (D * 1000).astype(int).tolist()
    t0 = time.time()
    tour = elkai.solve_int_matrix(D_int)
    runtime = time.time() - t0
    cost = sum(D[tour[i]][tour[i+1]] for i in range(len(tour) - 1)) + D[tour[-1]][tour[0]]
    return cost, runtime, None, tour

def get_algo_pool_val(n):
    if n<=30:
        epochs, pop = 200,100
    elif n<=75:
        epochs, pop = 300,75
    elif n<=100:
        epochs, pop = 400,100
    elif n<=150:
        epochs, pop = 500,150
    else:
        epochs, pop = 800,200
        
    return {
        "OGA": lambda D, e=epochs, p=pop: run_mealpy_val(GA.OriginalGA(epoch=e, pop_size=p), D),
        "SA":  lambda D, e=epochs: run_mealpy_val(SA.OriginalSA(epoch=e, pop_size=10), D),
        "LK":  lambda D: run_lk_val(D),
        "PSO": lambda D, e=epochs, p=pop: run_mealpy_val(PSO.OriginalPSO(epoch=e, pop_size=p), D),
        "ACO": lambda D, e=epochs, p=pop: run_mealpy_val(ACOR.OriginalACOR(epoch=e, pop_size=p), D),
    }
def test_solvers(n=20, distribution='uniform',seed=0):
    coords, D = generate_instance(n, distribution, seed)
    pool = get_algo_pool_val(n)

    print(f"\nTesting  on {distribution} instance, n={n}")
    print('-'*50)

    for name, algo_fn in pool.items():
        cost, runtime, _, tour = algo_fn(D)
        is_valid, msg = validate_tour(tour,n)
        print(f"{name:6} | valid={is_valid} | cost={cost:.2f} | runtime={runtime}| {msg}")
        
test_solvers(n=20, distribution="uniform", seed=0)
test_solvers(n=20, distribution="clustered", seed=0)
test_solvers(n=20, distribution="grid", seed=0)
test_solvers(n=50, distribution="diagonal", seed=0)


Testing  on uniform instance, n=20
--------------------------------------------------
OGA    | valid=True | cost=4364.42 | runtime=0.03130507469177246| Valid
SA     | valid=True | cost=4589.33 | runtime=0.019237518310546875| Valid
LK     | valid=True | cost=4364.42 | runtime=0.02635359764099121| Valid
PSO    | valid=True | cost=4393.80 | runtime=0.7166197299957275| Valid
ACO    | valid=True | cost=4474.14 | runtime=3.1805739402770996| Valid

Testing  on clustered instance, n=20
--------------------------------------------------
OGA    | valid=True | cost=1300.60 | runtime=0.024270296096801758| Valid
SA     | valid=True | cost=1331.30 | runtime=0.0165102481842041| Valid
LK     | valid=True | cost=1296.00 | runtime=0.03519582748413086| Valid
PSO    | valid=True | cost=1296.00 | runtime=0.7209348678588867| Valid
ACO    | valid=True | cost=1302.16 | runtime=1.5157718658447266| Valid

Testing  on grid instance, n=20
--------------------------------------------------
OGA    | valid=True | c

In [ ]:
train_configs = [
    {"n": n, "distribution": d, "seed": s}
    for n in [20,30,50,75,100,150,200]
    for d in ["uniform","clustered", "grid","diagonal"]
    for s in range(15)
    for dt in ["euclidean","manhattan","ceil_euclidean","chebyshev"]
]

TSPLIB_PROBLEMS = [
    "eil51", "berlin52", "st70", "eil76",
    "pr76", "kroA100", "eil101", "lin105"
]
val_configs = [{"name": p} for p in TSPLIB_PROBLEMS]
train_df = build_dataset(train_configs, source="generated", use_concorde=False)
val_df = build_dataset(val_configs, source="tsplib", use_concorde=True)

train_df.to_csv("train_meta.csv", index=False)
val_df.to_csv("val_meta.csv", index=False)

[1/1680] uniform_20_0 -> best: OGA
[2/1680] uniform_20_0 -> best: OGA
[3/1680] uniform_20_0 -> best: LK
[4/1680] uniform_20_0 -> best: SA
[5/1680] uniform_20_1 -> best: OGA
[6/1680] uniform_20_1 -> best: SA
[7/1680] uniform_20_1 -> best: ACO
[8/1680] uniform_20_1 -> best: OGA
[9/1680] uniform_20_2 -> best: LK
[10/1680] uniform_20_2 -> best: LK
[11/1680] uniform_20_2 -> best: LK
[12/1680] uniform_20_2 -> best: LK
[13/1680] uniform_20_3 -> best: SA
[14/1680] uniform_20_3 -> best: OGA
[15/1680] uniform_20_3 -> best: SA
[16/1680] uniform_20_3 -> best: ACO
[17/1680] uniform_20_4 -> best: OGA
[18/1680] uniform_20_4 -> best: SA
[19/1680] uniform_20_4 -> best: PSO
[20/1680] uniform_20_4 -> best: OGA
[21/1680] uniform_20_5 -> best: ACO
[22/1680] uniform_20_5 -> best: OGA
[23/1680] uniform_20_5 -> best: ACO
[24/1680] uniform_20_5 -> best: PSO
[25/1680] uniform_20_6 -> best: LK
[26/1680] uniform_20_6 -> best: LK
[27/1680] uniform_20_6 -> best: LK
[28/1680] uniform_20_6 -> best: LK
[29/1680] unifo

/tmp/ipykernel_10406/4084398841.py:25: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  "skew_nn": float(skew(nn)),


[121/1680] grid_20_0 -> best: OGA
[122/1680] grid_20_0 -> best: LK
[123/1680] grid_20_0 -> best: SA
[124/1680] grid_20_0 -> best: OGA
[125/1680] grid_20_1 -> best: SA
[126/1680] grid_20_1 -> best: OGA
[127/1680] grid_20_1 -> best: SA
[128/1680] grid_20_1 -> best: SA
[129/1680] grid_20_2 -> best: OGA
[130/1680] grid_20_2 -> best: LK
[131/1680] grid_20_2 -> best: OGA
[132/1680] grid_20_2 -> best: LK
[133/1680] grid_20_3 -> best: SA
[134/1680] grid_20_3 -> best: OGA
[135/1680] grid_20_3 -> best: SA
[136/1680] grid_20_3 -> best: OGA
[137/1680] grid_20_4 -> best: SA
[138/1680] grid_20_4 -> best: OGA
[139/1680] grid_20_4 -> best: OGA
[140/1680] grid_20_4 -> best: LK
[141/1680] grid_20_5 -> best: OGA
[142/1680] grid_20_5 -> best: LK
[143/1680] grid_20_5 -> best: OGA
[144/1680] grid_20_5 -> best: SA
[145/1680] grid_20_6 -> best: OGA
[146/1680] grid_20_6 -> best: OGA
[147/1680] grid_20_6 -> best: LK
[148/1680] grid_20_6 -> best: OGA
[149/1680] grid_20_7 -> best: LK
[150/1680] grid_20_7 -> best:

In [25]:
import mealpy
print(mealpy.__version__)


3.0.3


In [ ]:
from mealpy.utils.problem import Problem
import inspect
print(inspect.signature(Problem.__init__))

In [14]:
import mealpy.evolutionary_based.GA as GA_module
print(dir(GA_module))

['BaseGA', 'EliteMultiGA', 'EliteSingleGA', 'MultiGA', 'Optimizer', 'OriginalGA', 'SingleGA', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'np']
